In [ ]:
import os, glob, shutil, subprocess

MODEL_DIR = "/mnt/tmp"
REPO = "Comfy-Org/MiniMax-H3"
FILES = [
    "diffusion_models/minimax_h3_ref2va_bf16.safetensors",    # 61.7 GB
    "text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors",  # 48.0 GB
    "vae/minimax_h3_video_vae_fp16.safetensors",              # 4.9 GB
    "vae/minimax_h3_audio_vae_fp32.safetensors",              # 0.6 GB
]

# 1. 安装 modelscope(没装的话),并修复之前被升级的 huggingface-hub 版本冲突
subprocess.run('pip install -q modelscope "huggingface-hub>=1.5,<2.0"', shell=True)

# 2. 空间检查
os.makedirs(MODEL_DIR, exist_ok=True)
free = shutil.disk_usage(MODEL_DIR).free / 2**30
print(f"{MODEL_DIR} 可用 {free:.0f} GB")
assert free > 130, f"{MODEL_DIR} 空间不足 130GB"

# 3. 下载(实时输出进度)
cmd = ["modelscope", "download", "--model", REPO, *FILES, "--local_dir", MODEL_DIR]
print(" ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line.rstrip()[-200:])
p.wait()
print("下载返回码:", p.returncode)

# 4. 软链接到 ComfyUI
COMFY = "/mnt/workspace/ComfyUI"
for f in sorted(glob.glob(f"{MODEL_DIR}/**/*.safetensors", recursive=True)):
    n = os.path.basename(f)
    sub = ("diffusion_models" if "ref2va" in n else
           "text_encoders" if "qwen3vl" in n else
           "vae" if "vae" in n else None)
    if not sub: continue
    dst = f"{COMFY}/models/{sub}"; os.makedirs(dst, exist_ok=True)
    link = f"{dst}/{n}"
    if os.path.islink(link) or os.path.exists(link): os.remove(link)
    os.symlink(f, link)
    print(f"{os.path.getsize(f)/2**30:6.1f} GB  {sub}/{n}")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


/mnt/tmp 可用 558 GB
modelscope download --model Comfy-Org/MiniMax-H3 diffusion_models/minimax_h3_ref2va_bf16.safetensors text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors vae/minimax_h3_video_vae_fp16.safetensors vae/minimax_h3_audio_vae_fp32.safetensors --local_dir /mnt/tmp

 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  ._

In [5]:
import os, glob, subprocess

MODEL_DIR, COMFY = "/mnt/tmp", "/mnt/workspace/ComfyUI"
LORA = "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"

p = subprocess.Popen(["modelscope", "download", "--model", "Comfy-Org/MiniMax-H3", LORA,
                      "--local_dir", MODEL_DIR],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line.rstrip()[-200:])
p.wait(); print("返回码:", p.returncode)

for f in glob.glob(f"{MODEL_DIR}/**/*turbo_4step*.safetensors", recursive=True):
    dst = f"{COMFY}/models/loras"; os.makedirs(dst, exist_ok=True)
    link = f"{dst}/{os.path.basename(f)}"
    if os.path.islink(link) or os.path.exists(link): os.remove(link)
    os.symlink(f, link)
    print(f"{os.path.getsize(f)/2**30:.2f} GB  loras/{os.path.basename(f)}")


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'

ub/_download.py:1128: FutureWarning: Environment vari

In [2]:
!df -h

Filesystem                                                               Size  Used Avail Use% Mounted on
9e280ae6b495272a77046ec9b6d21a8de38d32d54d638cbf8397a6a79d1c5b4b-rootfs  591G  2.4G  559G   1% /
tmpfs                                                                     64M     0   64M   0% /dev
tmpfs                                                                    104G     0  104G   0% /sys/fs/cgroup
/dev/vda                                                                  30G   46M   28G   1% /tmp
tmpfs                                                                    104G  128K  104G   1% /etc/hosts
tmpfs                                                                     16G  4.0K   16G   1% /dev/shm
overlay                                                                   30G   46M   28G   1% /etc/dsw
172.31.1.151:/c46ed2c9-717d-4072-8fa3-d0b6519d2ef6                       1.0P   37T  988T   4% /mnt/data
virtiofs-default                                                    